Imports 

In [5]:
import numpy as np
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt
import plotly.express as px
from torchdiffeq import odeint
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import csv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Used device: {device}")

Used device: cuda


Physics

In [12]:
# Global court variables
BASKET_X, BASKET_Y, BASKET_Z = 1.575, 7.5, 3.05
BASKET_POS = torch.tensor([BASKET_X, BASKET_Y, BASKET_Z], dtype=torch.float32).to(device)
BASKET_RADIUS = 0.2285

def ball_equation(t, s):
    # Ensure input is always 2D (e.g., shape [32, 6] for 32 balls)
    if s.dim() == 1:
        s = s.unsqueeze(0)

    velocities = s[:, 3:]
    # Calculate speed for all balls at once
    v_value = torch.norm(velocities, dim=1, keepdim=True)

    g = torch.tensor([[0.0, 0.0, -9.807]], dtype=torch.float32).to(device)
    C_d, m, rho, A = 0.54, 0.624, 1.2, 0.0472
    k = (C_d * rho * A) / (2 * m)

    # Physics for all balls calculated in one line
    dv = g - k * v_value * velocities
    out = torch.cat([velocities, dv], dim=1)

    return out.squeeze(0) if out.shape[0] == 1 else out

def count_loss(v_preds, positions_0):
    # v_preds and positions now have shape [Batch, 3]
    batch_size = v_preds.shape[0]

    v_max = 25.0
    alpha, beta, gamma = 25.0, 0.5, -1.0        # alpha - dist loss | beta - velocity loss | angle - loss
    invalid_shot_penalty, max_v_penalty = 1000.0, 10000.0    # additional penalties

    # Create the starting matrix for all shots
    s0 = torch.cat([positions_0, v_preds], dim=1)
    t_span = torch.linspace(0, 4, 500, dtype=torch.float32).to(device)  # for 0,4,500 -> step=8ms

    # One call solves 32 differential equations in parallel
    trajectories = odeint(ball_equation, s0, t_span) # Shape: [t_span, Batch, 6]

    total_loss_batch = dist_loss_batch = vel_loss_batch = angle_loss_batch = 0.0

    # Extract loss for each computed trajectory
    for i in range(batch_size):
        traj = trajectories[:, i, :] # Extract i-th shot
        
        ground_hits = (traj[:, 2] <= 0).nonzero(as_tuple=True)[0]
        if len(ground_hits) > 0:
            traj = traj[:ground_hits[0]]

        positions = traj[:, :3]
        velocities = traj[:, 3:]

        valid_shot = False
        dist_loss = torch.tensor(0.0).to(device)
        angle_loss = torch.tensor(0.0).to(device)

        above_rim = torch.where(positions[:, 2] > BASKET_Z)
        if len(above_rim[0]) > 0:
            last_above_rim = above_rim[0][-1]
            if last_above_rim < len(positions) - 1:
                v_at_rim = velocities[last_above_rim]

                if v_at_rim[2] >= 0: # Penalty for a shot from below 
                    valid_shot = False
                else:
                    valid_shot = True
                    z_last_above = positions[last_above_rim, 2]
                    z_first_below = positions[last_above_rim + 1, 2]

                    fraction = (z_last_above - BASKET_Z) / (z_last_above - z_first_below)
                    target_x = positions[last_above_rim, 0] + fraction * (positions[last_above_rim + 1, 0] - positions[last_above_rim, 0])
                    target_y = positions[last_above_rim, 1] + fraction * (positions[last_above_rim + 1, 1] - positions[last_above_rim, 1])

                    distance_2D = torch.sqrt((target_x - BASKET_X)**2 + (target_y - BASKET_Y)**2)
                    dist_loss = alpha * distance_2D

                    v_value_at_rim = torch.linalg.norm(v_at_rim)
                    entry_angle = torch.abs(torch.arcsin(torch.clamp(v_at_rim[2]/v_value_at_rim, -1.0, 1.0)))
                    angle_loss = gamma * entry_angle

        if not valid_shot:
            distances_3D = torch.linalg.norm(positions - BASKET_POS, dim=1)
            dist_loss = torch.min(distances_3D) * invalid_shot_penalty

        v_value = torch.linalg.norm(v_preds[i])
        velocity_loss = beta * v_value
        overspeed_penalty = max_v_penalty * torch.relu(v_value - v_max)

        shot_loss = dist_loss + angle_loss + velocity_loss + overspeed_penalty

        total_loss_batch += shot_loss
        dist_loss_batch += dist_loss
        vel_loss_batch += velocity_loss
        angle_loss_batch += angle_loss

    # Return averaged results for the entire batch
    return (total_loss_batch / batch_size, dist_loss_batch / batch_size,
            vel_loss_batch / batch_size, angle_loss_batch / batch_size)

-------------------------
Saving loss data

In [13]:
def save_data(loss_history,path):
  total_loss_data = loss_history['total']
  dist_loss_data = loss_history['dist']
  vel_loss_data = loss_history['vel']
  angle_loss_data = loss_history['angle']

  data_loss = [[],[],[],[]]
  data_loss[0] = total_loss_data
  data_loss[1] = dist_loss_data
  data_loss[2] = vel_loss_data
  data_loss[3] = angle_loss_data

  numpy_data_loss = np.array(data_loss).T

  with open(path, 'w', newline='') as csvfile:
    row_0 = ['total', 'dist', 'vel', 'angle']
    writer = csv.writer(csvfile)
    writer.writerow(row_0)
    writer.writerows(numpy_data_loss)

Model


In [14]:
class BasketballShotNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        return self.net(x)

    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(self.state_dict(), path)
        print(f"Model saved to {path}")


Generating the dataset

In [15]:
def generate_batch(batch_size=256, random_z=True, const_z=2.11):

  data = []
  min_dist_xy = 1.5 * BASKET_RADIUS

  while len(data) < batch_size:
      x0 = np.random.uniform(0.0, 28.0)
      y0 = np.random.uniform(0.0, 15.0)
      dist_xy = np.sqrt((x0 - BASKET_X)**2 + (y0 - BASKET_Y)**2)
      
      if dist_xy >= min_dist_xy:
          if random_z:
            z0 = np.random.uniform(2.1, 2.5)
          else:
            z0 = const_z
          data.append([x0, y0, z0])

  return torch.tensor(data, dtype=torch.float32).to(device)

Training

In [ ]:
import torch.optim as optim

model = BasketballShotNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 300
batch_size = 2
batches_per_epoch = 2

history = {'total': [], 'dist': [], 'vel': [], 'angle': []}

print("Starting optimized training with DYNAMIC data generation and CENTERED coords...")
model.train()

for epoch in range(epochs):
    epoch_total = epoch_dist = epoch_vel = epoch_angle = 0.0

    for _ in tqdm(range(batches_per_epoch), desc=f"Epoch {epoch+1}/{epochs}", leave=False):

        batch = generate_batch(batch_size=batch_size, random_z=True)

        optimizer.zero_grad()

        predictions = model(batch)

        loss, d_loss, v_loss, a_loss = count_loss(predictions, batch)

        loss.backward()
        optimizer.step()

        epoch_total += loss.item()
        epoch_dist += d_loss.item()
        epoch_vel += v_loss.item()
        epoch_angle += a_loss.item()

    history['total'].append(epoch_total / batches_per_epoch)
    history['dist'].append(epoch_dist / batches_per_epoch)
    history['vel'].append(epoch_vel / batches_per_epoch)
    history['angle'].append(epoch_angle / batches_per_epoch)

    if (epoch + 1) % 5 == 0:
        avg_vel_5 = sum(history['vel'][-5:]) / 5
        avg_angle_5 = sum(history['angle'][-5:]) / 5
        avg_total_5 = sum(history['total'][-5:]) / 5
        model.save(path=f"models/model_epoch_{epoch+1}.pth")
        save_data(history.copy(), path=f"loss_data/loss_history_data_epoch_{epoch+1}.csv")
        print(f"Epoch [{epoch+1:3d}/{epochs}] | Average Loss -> Velocity: {avg_vel_5:.4f} | Angle: {avg_angle_5:.4f} | Total: {avg_total_5:.2f}")

print("Training finished!")

In [19]:
def is_shot_successful(v_pred, x_0, y_0, z_0, tolerance=0.1055):
    device = v_pred.device
    r = torch.tensor([x_0, y_0, z_0], dtype=torch.float32).to(device)
    s0 = torch.cat([r, v_pred])
    t_span = torch.linspace(0, 4, 500, dtype=torch.float32).to(device)

    with torch.no_grad():
        trajectory = odeint(ball_equation, s0, t_span)

    ground_hits = (trajectory[:, 2] <= 0).nonzero(as_tuple=True)[0]
    if len(ground_hits) > 0:
        trajectory = trajectory[:ground_hits[0]]

    positions = trajectory[:, :3]
    velocities = trajectory[:, 3:]

    above_rim = torch.where(positions[:, 2] > BASKET_Z)
    if len(above_rim[0]) > 0:
        last_above_rim = above_rim[0][-1]

        if last_above_rim < len(positions) - 1 and velocities[last_above_rim, 2] < 0:
            z_last_above_rim = positions[last_above_rim, 2]
            z_first_below_rim = positions[last_above_rim + 1, 2]

            fraction = (z_last_above_rim - BASKET_Z) / (z_last_above_rim - z_first_below_rim)
            target_x = positions[last_above_rim, 0] + fraction * (positions[last_above_rim + 1, 0] - positions[last_above_rim, 0])
            target_y = positions[last_above_rim, 1] + fraction * (positions[last_above_rim + 1, 1] - positions[last_above_rim, 1])

            distance_2D = torch.sqrt((target_x - BASKET_X)**2 + (target_y - BASKET_Y)**2)
            if distance_2D <= tolerance:
                return True
    return False

In [ ]:
def evaluate_model_accuracy(model, num_test_shots=1000):
    """
    Tests the model accuracy on a completely new, randomly generated dataset
    using the same court constraints as the training set.
    """
    print(f"Generating {num_test_shots} new test samples...")
    X_test = generate_batch(num_test_shots)

    model.eval()
    success_count = 0

    print(f"Evaluating model on the new dataset...")

    with torch.no_grad():

        chunk_size = 100
        for i in range(0, num_test_shots, chunk_size):
            batch_X = X_test[i : i + chunk_size]

            v_preds = model(batch_X)

            for j in range(v_preds.shape[0]):
                if is_shot_successful(v_preds[j], batch_X[j, 0], batch_X[j, 1], batch_X[j, 2]):
                    success_count += 1

    accuracy_percent = (success_count / num_test_shots) * 100

    print("\n" + "="*30)
    print(f"TEST RESULTS")
    print("-" * 30)
    print(f"Total shots: {num_test_shots}")
    print(f"Successful:  {success_count}")
    print(f"Accuracy:    {accuracy_percent:.2f}%")
    print("="*30)

    return accuracy_percent

evaluate_model_accuracy(model, num_test_shots=1000)

In [ ]:
def evaluate_model_accuracy_at_lowest_z(model, num_test_shots=1000):
    print(f"Generating {num_test_shots} new test samples...")
    X_test = generate_batch(num_test_shots, random_z=False, const_z=2.11)

    model.eval()
    success_count = 0

    print(f"Evaluating model on the new dataset...")

    with torch.no_grad():

        chunk_size = 100
        for i in range(0, num_test_shots, chunk_size):
            batch_X = X_test[i : i + chunk_size]

            v_preds = model(batch_X)

            for j in range(v_preds.shape[0]):
                if is_shot_successful(v_preds[j], batch_X[j, 0], batch_X[j, 1], batch_X[j, 2]):
                    success_count += 1

    accuracy_percent = (success_count / num_test_shots) * 100

    print("\n" + "="*30)
    print(f"TEST RESULTS at z=2.11")
    print("-" * 30)
    print(f"Total shots: {num_test_shots}")
    print(f"Successful:  {success_count}")
    print(f"Accuracy at z=2.11:    {accuracy_percent:.2f}%")
    print("="*30)

    return accuracy_percent

evaluate_model_accuracy_at_lowest_z(model, num_test_shots=1000)

Generating 1000 new test samples...
Evaluating model on the new dataset...

TEST RESULTS at z=2.11
------------------------------
Total shots: 1000
Successful:  872
Accuracy at z=2.11:    87.20%


87.2

In [ ]:
def evaluate_model_accuracy_at_highest_z(model, num_test_shots=1000):
    print(f"Generating {num_test_shots} new test samples...")
    X_test = generate_batch(num_test_shots, random_z=False, const_z=2.49)

    model.eval()
    success_count = 0

    print(f"Evaluating model on the new dataset...")

    with torch.no_grad():

        chunk_size = 100
        for i in range(0, num_test_shots, chunk_size):
            batch_X = X_test[i : i + chunk_size]

            v_preds = model(batch_X)

            for j in range(v_preds.shape[0]):
                if is_shot_successful(v_preds[j], batch_X[j, 0], batch_X[j, 1], batch_X[j, 2]):
                    success_count += 1

    accuracy_percent = (success_count / num_test_shots) * 100

    print("\n" + "="*30)
    print(f"TEST RESULTS at z=2.49")
    print("-" * 30)
    print(f"Total shots: {num_test_shots}")
    print(f"Successful:  {success_count}")
    print(f"Accuracy at z=2.49:    {accuracy_percent:.2f}%")
    print("="*30)

    return accuracy_percent

model.to(device)
evaluate_model_accuracy_at_highest_z(model, num_test_shots=1000)

Generating 1000 new test samples...
Evaluating model on the new dataset...

TEST RESULTS at z=2.49
------------------------------
Total shots: 1000
Successful:  961
Accuracy at z=2.49:    96.10%


96.1

In [ ]:
model = BasketballShotNet()
model.load_state_dict(torch.load("models/best_model.pth",weights_only=True,map_location=torch.device('cuda')))


In [ ]:
import plotly.express as px

def generate_and_plot_heatmap_fast(model, x_zones=100, y_zones=50, shots_per_zone=20, tolerance=0.1055):
    dx = 28.0 / x_zones
    dy = 15.0 / y_zones
    accuracy_matrix = np.zeros((x_zones, y_zones))

    print("Testing court zones in batches...")
    model.eval()

    t_span = torch.linspace(0, 4, 500, dtype=torch.float32).to(device)

    for i in tqdm(range(x_zones)):
        x_min, x_max = i * dx, (i + 1) * dx

        batch = []

        for j in range(y_zones):
            y_min, y_max = j * dy, (j + 1) * dy
            for _ in range(shots_per_zone):
                x_0 = np.random.uniform(x_min, x_max)
                y_0 = np.random.uniform(y_min, y_max)
                z_0 = np.random.uniform(1.8, 1.9)

                batch.append([x_0, y_0, z_0])

        batch = torch.tensor(batch, dtype=torch.float32).to(device)

        with torch.no_grad():
            v_preds = model(batch)
            s0 = torch.cat([batch, v_preds], dim=1)
            trajectories = odeint(ball_equation, s0, t_span, method='rk4', options={'step_size': 0.05})

        shot_idx = 0
        for j in range(y_zones):
            successful_shots = 0

            for _ in range(shots_per_zone):
                traj = trajectories[:, shot_idx, :]

                ground_hits = (traj[:, 2] <= 0).nonzero(as_tuple=True)[0]
                if len(ground_hits) > 0:
                    traj = traj[:ground_hits[0]]

                positions = traj[:, :3]
                velocities = traj[:, 3:]

                above_rim = torch.where(positions[:, 2] > BASKET_Z)
                if len(above_rim[0]) > 0:
                    last_above_rim = above_rim[0][-1]

                    if last_above_rim < len(positions) - 1 and velocities[last_above_rim, 2] < 0:
                        z_last_above_rim = positions[last_above_rim, 2]
                        z_first_below_rim = positions[last_above_rim + 1, 2]

                        fraction = (z_last_above_rim - BASKET_Z) / (z_last_above_rim - z_first_below_rim)
                        target_x = positions[last_above_rim, 0] + fraction * (positions[last_above_rim + 1, 0] - positions[last_above_rim, 0])
                        target_y = positions[last_above_rim, 1] + fraction * (positions[last_above_rim + 1, 1] - positions[last_above_rim, 1])

                        distance_2D = torch.sqrt((target_x - BASKET_X)**2 + (target_y - BASKET_Y)**2)

                        if distance_2D <= tolerance:
                            successful_shots += 1

                shot_idx += 1

            accuracy_matrix[i, j] = successful_shots / shots_per_zone

    fig = px.imshow(
        accuracy_matrix.T,
        labels=dict(x="Court width (X)", y="Court length (Y)", color="Accuracy"),
        x=np.linspace(0, 28, accuracy_matrix.shape[0]),
        y=np.linspace(0, 15, accuracy_matrix.shape[1]),
        color_continuous_scale="RdYlGn",
        origin='lower'
    )
    fig.update_layout(title="Model's Accuracy Heatmap")
    fig.show()

    return accuracy_matrix
np.random.seed(67)
accuracy_matrix=generate_and_plot_heatmap_fast(model, x_zones=100, y_zones=50, shots_per_zone=20)

In [ ]:
fig = px.imshow(
        accuracy_matrix.T,
        labels=dict(x="Court width (X)", y="Court length (Y)", color="Accuracy"),
        x=np.linspace(0, 28, accuracy_matrix.shape[0]),
        y=np.linspace(0, 15, accuracy_matrix.shape[1]),
        color_continuous_scale="RdYlGn",
        origin='lower'
)
fig.update_layout(title="Model's Accuracy Heatmap")
fig.update_traces(zsmooth='best')
fig.show()

In [ ]:
import plotly.graph_objects as go

print(accuracy_matrix.shape)

fig = go.Figure(data=go.Contour(
    z=accuracy_matrix.T,
    x=np.linspace(0, 28, accuracy_matrix.shape[0]),
    y=np.linspace(0, 15, accuracy_matrix.shape[1]),
    colorscale="RdYlGn",
    line_width=0.2,
    contours=dict(
        coloring='heatmap',
        showlines=False
    )
))

fig.update_layout(
    title="Model's Accuracy Heatmap",
    xaxis_title="Court width (X)",
    yaxis_title="Court length (Y)"
)
fig.show()

In [ ]:
def save_matrix(data, path="heatmap_data/matrix_data.csv"):

  with open(path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(data)

save_matrix(accuracy_matrix)